# HATS DEL — Interaktywna wizualizacja

Symulacja problemu kapeluszy w Dynamicznej Logice Epistemicznej.

**Lewy panel**: graf agentów (kolory ujawniane gdy agent odgaduje)  
**Prawy panel**: macierz możliwych światów per agent (jasnoniebieski = możliwy po obserwacji i komunikacji)

> Najpierw uruchom komórkę **Setup**, potem **Rysowanie**, potem **Widgety**.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path("build").resolve()))
warnings.filterwarnings("ignore", category=DeprecationWarning)

import hats_py
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

matplotlib.rcParams["figure.dpi"] = 90
%matplotlib inline

def make_graph(topology, n):
    if topology == "complete": return hats_py.Graph.make_complete(n)
    if topology == "cycle":    return hats_py.Graph.make_cycle(n)
    return hats_py.Graph.make_star(n)

def nx_from_hats(g, n):
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for u in range(n):
        for v in range(u + 1, n):
            if g.has_edge(u, v):
                G.add_edge(u, v)
    return G

print("hats_py loaded ✓  ipywidgets", widgets.__version__)


hats_py loaded ✓  ipywidgets 8.1.8


## Funkcja rysowania

In [2]:
# n <= MATRIX_N_MAX  → rysuj pełną macierz; powyżej → tylko tekst
# n <= LABEL_N_MAX   → rysuj etykiety kolumn (binarne); powyżej → bez etykiet
# Próg wyznaczony z benchmarku: 1023 etykiet = ~700 ms/klatkę → nieakceptowalne.
MATRIX_N_MAX = 9   # 2^9  =  512 kolumn — nadal szybko (~30 ms)
LABEL_N_MAX  = 6   # 2^6  =   63 kolumn — etykiety czytelne

def draw_simulation_frame(frame_idx, trace, g, ks, parts, n, actual_world):
    rec = trace[frame_idx]

    # cumulative guessed colors up to this frame
    guessed = [-1] * n
    for r in trace[:frame_idx + 1]:
        for j, p in enumerate(r.guessed_players):
            guessed[p] = r.guessed_colors[j]

    actual_hat = [(actual_world >> i) & 1 for i in range(n)]
    hat_str = ''.join('R' if c else 'B' for c in actual_hat)

    G = nx_from_hats(g, n)
    pos = nx.circular_layout(G) if n <= 6 else nx.spring_layout(G, seed=42)

    show_matrix = (n <= MATRIX_N_MAX)
    fig, axes = plt.subplots(1, 2, figsize=(13, max(4, min(n, 8))))

    # ── graf agentów ──────────────────────────────────────────────────
    ax_g = axes[0]
    node_colors = [
        'crimson'   if guessed[p] == 1 else
        'steelblue' if guessed[p] == 0 else
        'dimgray'
        for p in range(n)
    ]
    nx.draw_networkx(
        G, pos, ax=ax_g,
        node_color=node_colors, node_size=800,
        labels={v: str(v) for v in range(n)},
        font_color='white', font_weight='bold', edge_color='#aaaaaa',
    )
    legend_handles = [
        mpatches.Patch(color='crimson',   label='Czerwony (1)'),
        mpatches.Patch(color='steelblue', label='Biały (0)'),
        mpatches.Patch(color='dimgray',   label='Nieodgadnięty'),
    ]
    ax_g.legend(handles=legend_handles, loc='upper right', fontsize=8)
    ax_g.set_title(f'Graf agentów — runda {rec.round}\nŚwiat: {hat_str}', fontsize=10)
    ax_g.axis('off')

    # ── macierz lub tekst ─────────────────────────────────────────────
    ax_m = axes[1]
    if rec.silence:
        subtitle = f'Runda {rec.round} — cisza'
    elif rec.guessed_players:
        who = ', '.join(
            f'a{p}→{"R" if c else "B"}'
            for p, c in zip(rec.guessed_players, rec.guessed_colors)
        )
        subtitle = f'Runda {rec.round} — odgaduje: {who}'
    else:
        subtitle = f'Runda {rec.round}'

    if show_matrix:
        all_worlds = list(range(1, ks.world_count))
        valid_set = frozenset(rec.valid_worlds)

        # per-agent consistent worlds — numpy vectorized
        all_arr = np.array(all_worlds, dtype=np.int64)
        valid_mask = np.isin(all_arr, list(valid_set))
        mat = np.zeros((n, len(all_worlds)))
        for row in range(n):
            view = ks.actual_views[row]
            cls = parts[row].classes.get(view, [])
            view_mask = np.isin(all_arr, cls)
            mat[row] = (valid_mask & view_mask).astype(float)

        ax_m.imshow(mat, aspect='auto', cmap='Blues', vmin=0, vmax=1,
                    interpolation='nearest')
        ax_m.set_yticks(range(n))
        ax_m.set_yticklabels([f'Agent {i}' for i in range(n)], fontsize=9)
        ax_m.set_xlabel('Świat (bitmaska kapeluszy)', fontsize=9)

        if n <= LABEL_N_MAX:
            fmt = f'0{n}b'
            ax_m.set_xticks(range(len(all_worlds)))
            ax_m.set_xticklabels([format(w, fmt) for w in all_worlds],
                                   rotation=90, fontsize=7)
        else:
            # Tylko co 32. etykieta — czytelne i szybkie
            step = max(1, len(all_worlds) // 16)
            ticks = list(range(0, len(all_worlds), step))
            fmt = f'0{n}b'
            ax_m.set_xticks(ticks)
            ax_m.set_xticklabels([format(all_worlds[t], fmt) for t in ticks],
                                   rotation=90, fontsize=7)

        ax_m.set_xticks(np.arange(-0.5, len(all_worlds), 1), minor=True)
        ax_m.set_yticks(np.arange(-0.5, n, 1), minor=True)
        ax_m.grid(which='minor', color='white', linewidth=0.5)
        ax_m.tick_params(which='minor', bottom=False, left=False)
    else:
        # Dla dużego n: tekst zamiast macierzy (macierz miałaby 500+ kolumn)
        ax_m.axis('off')
        n_valid = len(rec.valid_worlds)
        n_total = ks.world_count - 1
        lines = [
            f'n = {n}  →  {n_total} światów łącznie',
            '',
            f'Ważnych światów: {n_valid} / {n_total}',
            f'Wyeliminowanych: {n_total - n_valid}',
            '',
            'Per agent (odgadnięty / kolor):',
        ]
        for p in range(n):
            status = f'{"R" if guessed[p]==1 else "B" if guessed[p]==0 else "—"}'
            lines.append(f'  Agent {p}: {status}')
        ax_m.text(0.05, 0.95, '\n'.join(lines), transform=ax_m.transAxes,
                  fontsize=10, va='top', fontfamily='monospace',
                  bbox=dict(boxstyle='round', facecolor='#e8f4f8', alpha=0.8))
        ax_m.set_xlim(0, 1); ax_m.set_ylim(0, 1)

    ax_m.set_title(subtitle, fontsize=10)

    n_guessed = sum(1 for x in guessed if x != -1)
    fig.suptitle(
        f'Klatka {frame_idx + 1}/{len(trace)}'
        f'  |  Odgadnięto: {n_guessed}/{n}'
        f'  |  Ważnych światów: {len(rec.valid_worlds)}',
        fontsize=10, y=1.01,
    )
    plt.tight_layout()
    plt.show()
    plt.close(fig)   # ← zapobiega kumulowaniu figur w pamięci


## Interaktywna symulacja

Wybierz parametry i kliknij **⚡ Uruchom**. Potem przechodź runda po rundzie przyciskami.

In [3]:
# ── state ──────────────────────────────────────────────────────────────
state = dict(trace=[], frame=0, g=None, ks=None, parts=None, n=3, world=0b011)

# ── controls ───────────────────────────────────────────────────────────
n_slider = widgets.IntSlider(
    value=3, min=3, max=12, step=1, description='n:',
    continuous_update=False, style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)
topo_dd = widgets.Dropdown(
    options=['complete', 'cycle', 'star'], value='complete',
    description='Topologia:', style={'description_width': 'initial'},
)
world_in = widgets.Text(
    value='0b011', description='Świat:', placeholder='0b..., 0x..., dec',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='200px'),
)
btn_run  = widgets.Button(description='⚡ Uruchom', button_style='success',
                           layout=widgets.Layout(width='140px'))
btn_prev = widgets.Button(description='◀ Poprzednia', disabled=True,
                           layout=widgets.Layout(width='140px'))
btn_next = widgets.Button(description='Następna ▶', disabled=True,
                           layout=widgets.Layout(width='140px'))
info_lbl = widgets.HTML('<i>Kliknij „Uruchom”, aby zacząć.</i>')
out      = widgets.Output()

# ── helpers ────────────────────────────────────────────────────────────
def refresh():
    n_frames = len(state['trace'])
    fi = state['frame']
    btn_prev.disabled = (fi == 0 or n_frames == 0)
    btn_next.disabled = (fi >= n_frames - 1 or n_frames == 0)
    with out:
        clear_output(wait=True)
        if state['trace']:
            draw_simulation_frame(
                fi, state['trace'],
                state['g'], state['ks'], state['parts'],
                state['n'], state['world'],
            )

def on_run(_):
    try:
        world = int(world_in.value, 0)
    except ValueError:
        info_lbl.value = "<b style='color:red'>Nieprawidłowy świat — wpisz np. 0b011 lub 3.</b>"
        return
    n = n_slider.value
    if world >= (1 << n):
        info_lbl.value = (
            f"<b style='color:red'>Świat {world} ≥ 2^{n} "
            f"— za duży dla {n} agentów!</b>"
        )
        return
    g     = make_graph(topo_dd.value, n)
    ks    = hats_py.init_knowledge(g, world)
    parts = hats_py.compute_partitions(g, ks.world_count)
    trace  = hats_py.Solver(g, world).trace()
    # Wynik wyciągamy z trace — nie potrzebujemy drugiego Solvera
    all_guessed = {p for r in trace for p in r.guessed_players}
    success = len(all_guessed) == n
    rounds  = trace[-1].round if trace else 0
    state.update(trace=trace, frame=0, g=g, ks=ks, parts=parts, n=n, world=world)
    status = '✅ Sukces' if success else '🔴 Deadlock'
    fmt = f'0{n}b'
    info_lbl.value = (
        f'<b>{status}</b> — {rounds} rund  |  '
        f'n={n}, topologia={topo_dd.value}, świat={format(world, fmt)}'
    )
    refresh()

def on_prev(_):
    state['frame'] = max(0, state['frame'] - 1)
    refresh()

def on_next(_):
    state['frame'] = min(len(state['trace']) - 1, state['frame'] + 1)
    refresh()

btn_run.on_click(on_run)
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)

# ── layout ─────────────────────────────────────────────────────────────
controls = widgets.VBox([
    widgets.HBox([n_slider, topo_dd, world_in]),
    widgets.HBox([btn_run, btn_prev, btn_next]),
    info_lbl,
])
display(controls, out)



Output()

## Porównanie topologii

In [4]:
# Szybkie porównanie topologii (n=3..8, świat = bity parzyste)
topologies = ['complete', 'cycle', 'star']

header = f"{'n':>3}  {'complete':>12}  {'cycle':>10}  {'star':>10}"
print(header)
print('─' * len(header))

for n in range(3, 9):
    world = sum(1 << i for i in range(0, n, 2))
    fmt = f'0{n}b'
    row = f'{n:>3} ({format(world, fmt)}) '
    for topo in topologies:
        g = make_graph(topo, n)
        r = hats_py.Solver(g, world).run()
        cell = f'✅ {r.rounds}r' if r.success else '🔴 dead'
        row += f' {cell:>12}'
    print(row)


  n      complete       cycle        star
─────────────────────────────────────────
  3 (101)          ✅ 2r         ✅ 2r       🔴 dead
  4 (0101)          ✅ 2r       🔴 dead       🔴 dead
  5 (10101)          ✅ 3r       🔴 dead       🔴 dead
  6 (010101)          ✅ 3r       🔴 dead       🔴 dead
  7 (1010101)          ✅ 4r       🔴 dead       🔴 dead
  8 (01010101)          ✅ 4r       🔴 dead       🔴 dead
